# Molecular Descriptors Calculator

Calculate RotB (Rotatable Bonds), tPSA (Topological Polar Surface Area), and LogP for a small set of compounds.

**Input format:** Dictionary with `ID: SMILES` pairs
**Output:** DataFrame with calculated descriptors

> **Note:** This notebook is standalone — it only imports RDKit directly, not the `py_utils` package.

In [1]:
# ═════════════════════════════════════════════════════════════════
# 1. DEFINE YOUR COMPOUNDS HERE (ID: SMILES)
# ═════════════════════════════════════════════════════════════════

compounds = {
    "A1C1O":    "CC(C)C1=NN(C(=O)C)C(=O)N1Cc1ccccc1",
    "A2C3S":    "CCOc1ccc(Cc2csc(NC(=O)C(C)(C)C)n2)cc1",
    "A5C7N":    "CCN(CC)C(=O)C1=NN(C(=O)c2ccccc2)C(=O)N1",
    "A10C15":   "O=C1N(Cc2ccccc2)C(=O)N(Cc2ccccc2)C1=O",
    "A20C25":   "O=C1NC(=O)N(Cc2ccccc2)C1=Cc1ccccc1",
    "A30C35":   "CCN(CC)C(=O)C1=NN(C)C(=O)N1Cc1ccc(O)cc1"
}

In [2]:
# ═════════════════════════════════════════════════════════════════
# 2. IMPORTS (RDKit only - no py_utils)
# ═════════════════════════════════════════════════════════════════

import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd

print(f"RDKit version: {rdkit.__version__}")
print(f"Compounds loaded: {len(compounds)}")

RDKit version: 2025.09.6
Compounds loaded: 6


In [3]:
# ═════════════════════════════════════════════════════════════════
# 3. CALCULATE DESCRIPTORS
# ═════════════════════════════════════════════════════════════════

def calculate_descriptors(smiles: str) -> dict:
    """
    Calculate RotB, tPSA, and LogP from a SMILES string.
    Returns dict with descriptors or None if SMILES is invalid.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Calculate descriptors
    RotB = Descriptors.NumRotatableBonds(mol)
    tPSA = Descriptors.TPSA(mol)
    LogP = Descriptors.MolLogP(mol)
    
    return {
        "RotB": RotB,
        "tPSA": tPSA,
        "LogP": LogP
    }


# Process all compounds
results = []

for comp_id, smiles in compounds.items():
    desc = calculate_descriptors(smiles)
    
    if desc is None:
        results.append({
            "ID": comp_id,
            "SMILES": smiles,
            "RotB": "INVALID",
            "tPSA": "INVALID",
            "LogP": "INVALID"
        })
    else:
        results.append({
            "ID": comp_id,
            "SMILES": smiles,
            "RotB": desc["RotB"],
            "tPSA": desc["tPSA"],
            "LogP": desc["LogP"]
        })

# Create DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*70)
print("MOLECULAR DESCRIPTORS CALCULATED")
print("="*70)
df_results


MOLECULAR DESCRIPTORS CALCULATED


,ID,SMILES,RotB,tPSA,LogP
0,A1C1O,CC(C)C1=NN(C(=O)C)C(=O)N1Cc1ccccc1,3,56.89,1.8766
1,A2C3S,CCOc1ccc(Cc2csc(NC(=O)C(C)(C)C)n2)cc1,5,51.22,4.1172
2,A5C7N,CCN(CC)C(=O)C1=NN(C(=O)c2ccccc2)C(=O)N1,4,88.06,0.7419
3,A10C15,O=C1N(Cc2ccccc2)C(=O)N(Cc2ccccc2)C1=O,4,57.69,2.1776
4,A20C25,O=C1NC(=O)N(Cc2ccccc2)C1=Cc1ccccc1,3,49.41,2.7795
5,A30C35,CCN(CC)C(=O)C1=NN(C)C(=O)N1Cc1ccc(O)cc1,5,80.36,0.8177


In [4]:
# ═════════════════════════════════════════════════════════════════
# 4. SUMMARY STATISTICS
# ═════════════════════════════════════════════════════════════════

# Filter only valid results for statistics
df_valid = df_results[df_results["RotB"] != "INVALID"]

print("\n" + "="*70)
print("SUMMARY STATISTICS (only valid compounds)")
print("="*70)

summary = df_valid.describe()
print(summary[["RotB", "tPSA", "LogP"]])

print("\n" + "="*70)
print(f"Total compounds: {len(compounds)}")
print(f"Valid compounds: {len(df_valid)}")
print(f"Invalid compounds: {len(df_results) - len(df_valid)}")
print("="*70)


SUMMARY STATISTICS (only valid compounds)
           RotB       tPSA      LogP
count  6.000000   6.000000  6.000000
mean   4.000000  63.938333  2.085083
std    0.894427  16.205401  1.270759
min    3.000000  49.410000  0.741900
25%    3.250000  52.637500  1.082425
50%    4.000000  57.290000  2.027100
75%    4.750000  74.692500  2.629025
max    5.000000  88.060000  4.117200

Total compounds: 6
Valid compounds: 6
Invalid compounds: 0
